# here is how we would train a sequence classifier on one batch:

In [32]:
import torch
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

sequences = [
    "I've been waiting for a HuggingFace course my whole life.",
    "This course is amazing!",
]

batch = tokenizer(sequences, padding=True, truncation=True, return_tensors='pt')

batch['labels'] = torch.tensor([1, 1])

optimizer = AdamW(model.parameters())
loss = model(**batch).loss
loss.backward()
optimizer.step()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 826.21it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

Of course, just training the model on two sentences is not going to yield very good results. To get better results, you will need to prepare a bigger dataset.

In [33]:
from datasets import load_dataset

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [34]:
from datasets import load_dataset_builder

builder = load_dataset_builder("nyu-mll/glue", "mrpc")
builder.info.features, builder.info.splits, builder.info.download_size, builder.config, builder.info.download_size

({'sentence1': Value('string'),
  'sentence2': Value('string'),
  'label': ClassLabel(names=['not_equivalent', 'equivalent']),
  'idx': Value('int32')},
 {'train': SplitInfo(name='train', num_bytes=944761, num_examples=3668, dataset_name='glue'),
  'validation': SplitInfo(name='validation', num_bytes=105981, num_examples=408, dataset_name='glue'),
  'test': SplitInfo(name='test', num_bytes=442842, num_examples=1725, dataset_name='glue')},
 1033400,
 ParquetConfig(name='mrpc', version=0.0.0, data_dir=None, data_files={'train': ['hf://datasets/nyu-mll/glue@bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/mrpc/train-00000-of-00001.parquet'], 'validation': ['hf://datasets/nyu-mll/glue@bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/mrpc/validation-00000-of-00001.parquet'], 'test': ['hf://datasets/nyu-mll/glue@bcdcba79d07bc864c1c254ccfcedcce55bcc9a8c/mrpc/test-00000-of-00001.parquet']}, description=None, batch_size=None, columns=None, features=None, filters=None, fragment_scan_options=None, on_bad_files='

In [35]:
raw_train_dataset = raw_datasets['train']

In [36]:
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [37]:
print(checkpoint)
print(type(raw_train_dataset["sentence1"]))
print(type(raw_train_dataset["sentence1"][0]))

print(raw_train_dataset["sentence1"][0])
print(raw_train_dataset["sentence2"][0])

bert-base-uncased
<class 'datasets.arrow_dataset.Column'>
<class 'str'>
Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .
Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .


In [38]:
def tokenize_function(example, tokenizer):
    return tokenizer(
        example['sentence1'],
        example['sentence2'],
        truncation=True
    )

In [39]:
tokenized_dataset = raw_train_dataset.map(tokenize_function, batched=True, fn_kwargs={"tokenizer": tokenizer})

In [40]:
tokenized_dataset.info.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32'),
 'input_ids': List(Value('int32')),
 'token_type_ids': List(Value('int8')),
 'attention_mask': List(Value('int8'))}

In [41]:
tokenized_dataset = tokenized_dataset.remove_columns(['idx', 'sentence1', 'sentence2']).rename_column('label', 'labels').with_format('torch')

In [42]:
from transformers import DataCollatorWithPadding
from torch.utils.data import DataLoader

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(tokenized_dataset, batch_size=16, shuffle=True, collate_fn=data_collator)

In [43]:
len(train_loader.dataset)

3668

In [44]:
train_loader.dataset[:5]

{'labels': tensor([1, 0, 1, 0, 1]),
 'input_ids': [tensor([  101,  2572,  3217,  5831,  5496,  2010,  2567,  1010,  3183,  2002,
           2170,  1000,  1996,  7409,  1000,  1010,  1997,  9969,  4487, 23809,
           3436,  2010,  3350,  1012,   102,  7727,  2000,  2032,  2004,  2069,
           1000,  1996,  7409,  1000,  1010,  2572,  3217,  5831,  5496,  2010,
           2567,  1997,  9969,  4487, 23809,  3436,  2010,  3350,  1012,   102]),
  tensor([  101,  9805,  3540, 11514,  2050,  3079, 11282,  2243,  1005,  1055,
           2077,  4855,  1996,  4677,  2000,  3647,  4576,  1999,  2687,  2005,
           1002,  1016,  1012,  1019,  4551,  1012,   102,  9805,  3540, 11514,
           2050,  4149, 11282,  2243,  1005,  1055,  1999,  2786,  2005,  1002,
           6353,  2509,  2454,  1998,  2853,  2009,  2000,  3647,  4576,  2005,
           1002,  1015,  1012,  1022,  4551,  1999,  2687,  1012,   102]),
  tensor([  101,  2027,  2018,  2405,  2019, 15147,  2006,  1996,  4274,  

base preprocessing(tokenization and padding with DataCollator)

In [4]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("nyu-mll/glue", "mrpc")
checkpoint = 'bert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [5]:
def tokenize_function(example, tokenizer):
    return tokenizer(
        example['sentence1'],
        example['sentence2'],
        truncation=True
    )

In [6]:
tokenized_dataset = raw_datasets.map(tokenize_function, batched=True, fn_kwargs={'tokenizer': tokenizer})
data_collator = DataCollatorWithPadding(tokenizer)

In [7]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 733.75it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint.

In [20]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    'test-trainer',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=20,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch'
)

In [21]:
from transformers import Trainer

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    data_collator=data_collator,
    processing_class=tokenizer
)

In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [13]:
predictions = trainer.predict(tokenized_dataset['validation'])
predictions.predictions.shape, predictions.label_ids.shape

((408, 2), (408,))

In [14]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print({
    "accuracy": accuracy_score(labels, preds),
    "f1": f1_score(labels, preds),
})

{'accuracy': 0.8455882352941176, 'f1': 0.8908145580589255}


In [23]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
    }

In [24]:
trainer.compute_metrics=compute_metrics

In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,1.047514,0.848039,0.889680
2,No log,1.065337,0.845588,0.889667
3,0.051359,1.204308,0.855392,0.899145
4,0.051359,1.153819,0.848039,0.891986
5,0.021817,1.269345,0.852941,0.897260
6,0.021817,1.205023,0.860294,0.901554
7,0.009529,1.424109,0.850490,0.898502
8,0.009529,1.350813,0.855392,0.900169
9,0.013615,0.933314,0.857843,0.898601
10,0.013615,1.425637,0.855392,0.899489


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.97it/s]


TrainOutput(global_step=4600, training_loss=0.012123582141230936, metrics={'train_runtime': 775.1639, 'train_samples_per_second': 94.638, 'train_steps_per_second': 5.934, 'total_flos': 2996312809549920.0, 'train_loss': 0.012123582141230936, 'epoch': 20.0})

In [26]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.000266,1.486853,20,0.852941,0.897260


{'eval_loss': 1.4868528842926025,
 'eval_accuracy': 0.8529411764705882,
 'eval_f1': 0.8972602739726028}

In [27]:
from transformers import EarlyStoppingCallback, set_seed

set_seed(42)

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
)

training_args = TrainingArguments(
    output_dir="bert-early-stopping",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2)
    ],
)

trainer.train()
trainer.evaluate()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 699.44it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint.

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.569556,0.427225,0.791667,0.859967
2,0.351659,0.394506,0.833333,0.883162
3,0.189955,0.467231,0.843137,0.880597
4,0.100260,0.729116,0.843137,0.891156


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.63it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.100260,0.394659,4,0.833333,0.883162


{'eval_loss': 0.3946593105792999,
 'eval_accuracy': 0.8333333333333334,
 'eval_f1': 0.8831615120274914}